In [1]:
puts `ls -lt ./maps/`

total 868
-rw-rw-r-- 1 osboxes osboxes  84435 Apr 18 09:53 2026-drug-mapping-errors.txt
-rw-rw-r-- 1 osboxes osboxes 103903 Apr 18 09:53 2026-drug-mappings.map
drwxrwxr-x 2 osboxes osboxes   4096 Apr 18 09:03 deprecated
-rw-rw-r-- 1 osboxes osboxes   4830 Apr 17 13:36 2026-gene-errors.txt
-rw-rw-r-- 1 osboxes osboxes 380756 Apr 17 13:36 2026-gene-mappings.map
-rw-rw-r-- 1 osboxes osboxes   8814 Apr 17 10:46 2026-unmapped-cuis.csv
-rw-rw-r-- 1 osboxes osboxes 275711 Apr 17 10:46 2026-demokritos-disease-mondo.map


In [2]:
puts `head -2 ./maps/2026-drug-mappings.map`

demokritosid,xref,pubchem_cid,IUPACname
C0000376,http://purl.bioontology.org/ontology/MESH/D015102,https://pubchem.ncbi.nlm.nih.gov/compound/547,"3,4-Dihydroxyphenylacetic Acid"


In [3]:
puts `head -2 ./maps/2026-gene-mappings.map`

source,label,geneid,protein,recommended_full,taxon
C0079419,TP53,http://purl.uniprot.org/geneid/7157,http://purl.uniprot.org/uniprot/P04637,tumor protein p53,http://purl.uniprot.org/taxonomy/9606


In [4]:
puts `head -2 "./raw-data/Drug-Gene triples.tsv"`
puts `cat "./raw-data/Drug-Gene triples.tsv" | wc -l`
puts
puts `head -2 "./raw-data/Gene-Drug triples.tsv"`
puts `cat "./raw-data/Gene-Drug triples.tsv" | wc -l`

Drug	Drug_id	RELATION	PROVENANCE	Gene	Gene_id
Lysine	C0024337	PART_OF	["36128717_fullText_114"]	LOR gene	C1416897
1028

Gene	Gene_id	RELATION	PROVENANCE	Drug	Drug_id
Alleles	C0002085	COEXISTS_WITH	["36128717_fullText_53"]	Lysine	C0024337
663


In [ ]:
require 'linkeddata'
require 'csv'

graphing_errors = File.open('./graph/2026-drug-gene-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

# Read input files
drug_mappings = CSV.read('./maps/2026-drug-mappings.map', headers: true)
gene_mappings = CSV.read('./maps/2026-gene-mappings.map', headers: true)

# Create RDF graph
graph = RDF::Repository.new

failures = {}
# Process each entity relation
recordcount = 0
['./raw-data/Drug-Gene triples.tsv', './raw-data/Gene-Drug triples.tsv'].each do |sourcefile|
      CSV.foreach(sourcefile, col_sep: "\t", quote_char: '"', liberal_parsing: true, headers: true) do |row|  # warn row.inspect
      print "#{recordcount}, " 
      recordcount=recordcount + 1
      
      # warn row.inspect
      drug_id = row['Drug_id']
      gene_id = row['Gene_id']
      evidence = row['PROVENANCE'] # ["36128717_fullText_53"]
      evidence_url = nil
      if evidence =~ /(\d+)_\w+_/
          evidence_url = "https://pubmed.ncbi.nlm.nih.gov/#{$1}"  # use short form for matches
      end
      source_relation = row['RELATION']
      # warn "Drug #{drug_id} -- Gene #{gene_id}"
      # Find corresponding mappings
      drug = drug_mappings.find { |d| d['demokritosid'] == drug_id }
      gene = gene_mappings.find { |d| d['source'] == gene_id }
      
      unless drug
        next if failures[drug_id]
        failures[drug_id] = 1
        # warn "drug lookup failed #{drug_id}"
        graphing_errors.write "drug lookup failed #{drug_id}\n"
        next
      end
      unless gene
        next if failures[gene_id]
        failures[gene_id] = 1
        # warn "gene lookup failed #{gene_id}"
        graphing_errors.write "gene lookup failed #{gene_id}\n"
        next
      end
      
      # Extract relevant IDs and labels
      # demokratisid,xref,demokratis_label,pubchem_cid,IUPACname
      # C0613621,http://purl.bioontology.org/ontology/MESH/C030536,"2,2-dichloro-1,1-difluoroethyl difluoromethyl ether",https://pubchem.ncbi.nlm.nih.gov/compound/152803,"2,2-dichloro-1,1-difluoroethyl%20difluoromethyl%20ether"
      pubchem_uri = RDF::URI.new(drug['pubchem_cid'])
      pubchem_type = RDF::URI.new("http://semanticscience.org/resource/CHEMINF_000302")
      pubchem_label =  RDF::Literal.new(drug['xref'])
      pubchem_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Drug")
      # human_drug_label = RDF::Literal.new(drug['demokritos_label'])
      iupac_drug_label = RDF::Literal.new(drug['IUPACname'])
    
      #   source,label,geneid,protein,recommended_full,taxon
      #   C1421313,UCP1,http://purl.uniprot.org/geneid/7350,http://purl.uniprot.org/uniprot/P25874,uncoupling protein 1,http://purl.uniprot.org/taxonomy/9606
      gene_uri = RDF::URI.new(gene['geneid'])
      gene_type = RDF::URI.new("http://edamontology.org/data_2610")
      gene_label =  RDF::Literal.new(gene['label'])
      gene_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Gene")
      human_gene_label = RDF::Literal.new(gene['recommended_full'])
    
      protein_uri = RDF::URI.new(gene['protein'])
      protein_type = RDF::URI.new("http://edamontology.org/data_2291")
      protein_label =  RDF::Literal.new(gene['protein'])
      protein_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Protein")
      human_protein_label = RDF::Literal.new(gene['recommended_full'])
    
      taxon = RDF::URI.new(gene['taxon'])
    
      # Create context URI
      general_context = RDF::URI.new("urn:simpathic:context:all_metadata")
      

      # Add quads to graph using RDF::Statement
      if sourcefile =~ /Gene\-Drug/
        context_uri = RDF::URI.new("urn:simpathic:context:dem_#{gene_id}_#{drug_id}")
          graph << RDF::Statement.new(gene_uri, SIMPATHIC['associated-with'], pubchem_uri, graph_name: context_uri)
      elsif sourcefile =~ /Drug\-Gene/
        context_uri = RDF::URI.new("urn:simpathic:context:dem_#{drug_id}_#{gene_id}")
          graph << RDF::Statement.new(pubchem_uri, SIMPATHIC['associated-with'], gene_uri, graph_name: context_uri)
      else
          abort "filename matching for directionality failed"
      end
      
      # graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     human_drug_label, graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     iupac_drug_label, graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_type,          graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_core_type,             graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_type, RDFS.label,     RDF::Literal.new("PubChem"), graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_core_type, RDFS.label,     RDF::Literal.new("Drug"), graph_name: context_uri)
      graph << RDF::Statement.new(pubchem_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{drug_id}"), graph_name: context_uri)
      
      graph << RDF::Statement.new(gene_uri,  RDFS.label,       human_gene_label , graph_name: context_uri)
      graph << RDF::Statement.new(gene_uri,  RDF.type,         gene_type, graph_name: context_uri)
      graph << RDF::Statement.new(gene_uri,  RDF.type,         gene_core_type, graph_name: context_uri)
      graph << RDF::Statement.new(gene_type, RDFS.label,       RDF::Literal.new("NCBI Gene"), graph_name: context_uri)
      graph << RDF::Statement.new(gene_core_type, RDFS.label,  RDF::Literal.new("Gene"), graph_name: context_uri)
      graph << RDF::Statement.new(gene_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{gene_id}"), graph_name: context_uri)
      graph << RDF::Statement.new(gene_uri,  SIMPATHIC['in-taxon'], taxon, graph_name: context_uri)
    
      
      
      graph << RDF::Statement.new(pubchem_uri, SIMPATHIC['associated-with'], protein_uri, graph_name: context_uri)
      graph << RDF::Statement.new(protein_uri, SIMPATHIC['associated-with'], pubchem_uri, graph_name: context_uri)
    
      graph << RDF::Statement.new(protein_uri,  RDFS.label,       human_protein_label , graph_name: context_uri)
      graph << RDF::Statement.new(protein_uri,  RDF.type,         protein_type, graph_name: context_uri)
      graph << RDF::Statement.new(protein_uri,  RDF.type,         protein_core_type, graph_name: context_uri)
      graph << RDF::Statement.new(protein_type, RDFS.label,       RDF::Literal.new("UniProt"), graph_name: context_uri)
      graph << RDF::Statement.new(protein_core_type, RDFS.label,  RDF::Literal.new("Protein"), graph_name: context_uri)
      graph << RDF::Statement.new(protein_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{gene_id}"), graph_name: context_uri)
      graph << RDF::Statement.new(protein_uri,  SIMPATHIC['in-taxon'], taxon, graph_name: context_uri)
    
      
      graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'], RDF::Literal.new("Demokritos"), graph_name: general_context)
          # evidence = row['PROVENANCE']
    # source_relation = row['RELATION']
    graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'], 
                                RDF::URI.new(evidence_url)) if evidence_url
    graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'],
                                RDF::Literal.new(source_relation))
    end
end

# Write RDF to file in N-Quads format
File.open('./graph/2026-demokritos_drug-gene.nq.large', 'w') do |f|
  RDF::Writer.for(:nquads).new(f) do |writer|
    writer << graph
  end
end
graphing_errors.close

puts "RDF quads written"

In [18]:
graphing_errors.close